# Chapter 28 — What Should Happen Next?

**Companion to *Applied AI*.**

Two questions are usually merged into one: *what* should happen next, and
*which model* should do it. This notebook keeps them apart. It replays the
preserved 16-case scheduler matrix — a deterministic, model-free rule for
the next operation — and the Stage 29B ladder decision, where the cheaper
climbing strategy failed its own adoption rule on one wrong acceptance.

## Question

**Which operation comes next — and did the cheaper ladder earn adoption?**

## What this notebook does

It **inspects** `scheduler/results.json` (16/16 code↔table parity, the
precedence chain, the dead ACTION path) and **reproduces** the ladder
verdict from `execution-ladder/2026-09-14-7a0d43b/`: cheaper per accepted
outcome, one more correct — and still `ladder_justified: false`.

```text
what operation next?  ≠  which model performs it?
```

## Setup

Standard library only. No network, no API key, no `codeai` import. Only
bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`.

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="scheduler"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
SCH = EVIDENCE_DIR / "scheduler"
LAD = EVIDENCE_DIR / "execution-ladder" / "2026-09-14-7a0d43b"
print("bundles: scheduler/")
print("         execution-ladder/2026-09-14-7a0d43b/")
sched = json.loads((SCH / "results.json").read_text(encoding="utf-8"))
lad = json.loads((LAD / "analysis.json").read_text(encoding="utf-8"))
matrix = sched["matrix_16"]
print("matrix rows:", len(matrix), "| parity all 16:", sched["parity_all_16"])

bundles: scheduler/
         execution-ladder/2026-09-14-7a0d43b/
matrix rows: 16 | parity all 16: True


## 1. The next operation follows a deterministic rule

`scheduler.decide_next_step` is pure: no model, no I/O. Four boolean inputs,
one operation out. The preserved matrix pins all 16 combinations with
code↔table parity. Reproduce the precedence chain locally and check it
against every frozen row — current state + evidence + policy → next
operation, with the rule that fired named:

In [2]:
def decide_next(budget_exhausted, has_required_verification,
                  requests_independent_proposals, requires_destructive):
    """The chapter's deterministic precedence: budget first, then
    verification before generation, then scheduled cognition, then a person.
    Scheduling cognition never authorizes a destructive action."""
    if budget_exhausted:
        return "STOP", "budget exhausted"
    if has_required_verification:
        return "CHECK", "required deterministic verification exists"
    if requests_independent_proposals:
        return "CALL", "task requests independent proposals"
    if requires_destructive:
        return "ASK_HUMAN", "destructive capability requires human"
    return "STOP", "no epistemic operation required"

flags = ("budget_exhausted", "has_required_verification",
         "requests_independent_proposals", "requires_destructive_capability")
ok = 0
for row in matrix:
    got = decide_next(*(row["input"][f] for f in flags))
    assert got[0] == row["code_operation"] == row["table_operation"], row["input"]
    assert row["parity"] is True
    ok += 1
print(f"local rule reproduces all {ok}/16 frozen rows (code and table).")
for finding in sched["key_precedence_findings"]:
    print(" -", finding)

local rule reproduces all 16/16 frozen rows (code and table).
 - budget+verification+proposals+destructive -> STOP (budget first)
 - verification+proposals+destructive -> CHECK (verification before generation)
 - proposals+destructive (no budget/verification flags) -> CALL: schedules cognition, does NOT authorize the destructive action
 - destructive alone -> ASK_HUMAN; nothing -> STOP


## 2. What the rule refuses to select

`ACTION` is dead: no ACTION-selection rule was added. Effects need explicit
authority plus a human gate (Chapters 20/29) — inventing a seam path to
ACTION would complete the enum cosmetically while bypassing the authority
story. The scheduler answers "what operation next", never "go change the
world":

In [3]:
print("operations reachable:", sched["operations_reachable"])
print("ACTION selectable     :", not sched["action_dead"])
print("verdict:", sched["action_verdict"][:160], "...")
assert sched["action_dead"] is True
assert set(sched["operations_reachable"]) == {"ASK_HUMAN", "CALL", "CHECK", "STOP"}
print()
print("A router that could schedule its own effects would be exactly the")
print("architecture the authority chapters refused to build.")

operations reachable: ['ASK_HUMAN', 'CALL', 'CHECK', 'STOP']
ACTION selectable     : False
verdict: No ACTION-selection rule added: effects need explicit authority + human gate (Ch20/29); inventing a seam path to ACTION would complete the enum cosmetically whi ...

A router that could schedule its own effects would be exactly the
architecture the authority chapters refused to build.


## 3. The cheaper ladder that failed its own rule

Stage 29B: a ladder of rule → free → cheap → strong rungs against sending
every item to the strong model first. Recompute the frozen verdict — the
ladder was cheaper per accepted outcome AND one more correct, yet not
adopted, because the rule required no *increase* in wrong acceptances:

In [4]:
dec = lad["decision"]
conds = dec["conditions"]
print("cheaper per accepted outcome      :", conds["cheaper_per_accepted_outcome"])
print("correct acceptances within one    :", conds["correct_acceptances_within_one"])
print("no more accepted-but-wrong        :", conds["no_more_negative_acceptances"])
print("ladder_justified                  :", dec["ladder_justified"])
print("verdict                           :", dec["verdict"])
assert conds["cheaper_per_accepted_outcome"] and conds["correct_acceptances_within_one"]
assert conds["no_more_negative_acceptances"] is False
assert dec["ladder_justified"] is False

# The cost column is a scenario ($5/person), executed with frozen figures:
lad_c = (0.0056 + 7 * 5) / 33
top_c = (0.0317 + 9 * 5) / 31
print(f"ladder cost/accepted: {lad_c:.3f} | top-first: {top_c:.3f}")
assert abs(lad_c - 1.061) < 0.005 and abs(top_c - 1.453) < 0.005
print()
print("One clause fired — the one written to protect exactly what it")
print("protected (A05, accepted and wrong). Cheaper, one more correct, and")
print("still refused. That is a decision rule doing its job, not a loss.")

cheaper per accepted outcome      : True
correct acceptances within one    : True
no more accepted-but-wrong        : False
ladder_justified                  : False
verdict                           : top-rung-first wins on this workload
ladder cost/accepted: 1.061 | top-first: 1.453

One clause fired — the one written to protect exactly what it
protected (A05, accepted and wrong). Cheaper, one more correct, and
still refused. That is a decision rule doing its job, not a loss.


## 4. The challenger that has not run

The book's standing bet — operation selection contains no model call — now
has a serious opponent: learned production routers that choose *which model*
in real time. That establishes learned routing for model choice, not for
operation choice. CodeAI's answer is an experiment *design* with a falsifier
(a model router ≥15 points better at operation selection, no stratum worse
by >5, no new catastrophes...), not a result. The design even states its own
power limit. No notebook cell can run an experiment that has not run:

In [5]:
print("supports    :", sched["supports"][:120], "...")
print("does_not_support:", sched["does_not_support"][:120], "...")
print()
print("PRESERVED NEGATIVE RESULT: the stronger abstraction — a model that")
print("routes operations, a policy-as-data router — did not earn promotion.")
print("This notebook builds the deterministic rule the evidence supports,")
print("not the architecture the experiment rejected.")

supports    : deterministic WHAT-next separated from WHICH-model; precedence under simultaneous conditions; reason+version on every de ...
does_not_support: autonomous loop (no loop exists); learned routing; budget/authority enforcement inside the seam (flags are precomputed;  ...

PRESERVED NEGATIVE RESULT: the stronger abstraction — a model that
routes operations, a policy-as-data router — did not earn promotion.
This notebook builds the deterministic rule the evidence supports,
not the architecture the experiment rejected.


## Interpretation

1. **Next-operation ≠ which-model.** A pure function answers the first from
   explicit state; learned routing is credible for the second and unproven
   for the first.
2. **Current state + evidence + policy → next operation**, with the fired
   rule named. Budget first; verification before generation; scheduled
   cognition authorizes nothing; destructive work asks a person.
3. **ACTION stays downstream-only.** No seam path schedules effects, because
   effects need authority the scheduler does not hold.
4. **Rules can refuse their own author.** The ladder met two of three
   adoption conditions and was rejected on the third — with the receipts.

## Try it yourself

1. Add a fifth flag, `has_stale_basis`, that forces CHECK before CALL when a
   cited decision's basis moved. Which matrix rows change? (Chapters 18–19
   say why this is load-bearing.)
2. Price the ladder table at $0 and $25 per person asked. The ordering holds;
   the absolutes are scenario artifacts. Show both.
3. Draft the model-router falsifier's first three cases. What would "15
   points better at operation selection" have to survive that accuracy
   alone does not show?

*Evidence: `experiments/applied-ai/evidence/scheduler/` (`results.json`,
16-case matrix + verifier) and
`experiments/applied-ai/evidence/execution-ladder/2026-09-14-7a0d43b/`
(`analysis.json` decision). No network, no API key, no `codeai` import.*